# Data Preparation with Processing Jobs

## What are Processing Jobs?

Processing Jobs run data processing scripts on managed infrastructure. They scale automatically, handle distributed processing, and integrate with S3 for input/output data. Unlike training jobs, they're optimized for data transformation rather than model training.

## SKLearnProcessor for Data Cleaning

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
import sagemaker

session = sagemaker.Session()
role = 'arn:aws:iam::123456789012:role/SageMakerRole'

# Create SKLearnProcessor
sklearn_processor = SKLearnProcessor(
    framework_version='0.23-1',
    role=role,
    instance_type='ml.m5.xlarge',
    instance_count=1,
    sagemaker_session=session
)

# Run processing job
sklearn_processor.run(
    code='preprocessing.py',
    inputs=[
        ProcessingInput(
            source='s3://my-bucket/raw-data/',
            destination='/opt/ml/processing/input'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination='s3://my-bucket/processed-data/'
        )
    ],
    arguments=['--input-data', '/opt/ml/processing/input']
)

## SparkProcessor for Distributed ETL

In [ ]:
from sagemaker.spark.processing import PySparkProcessor

spark_processor = PySparkProcessor(
    framework_version='2.4',
    role=role,
    instance_type='ml.m5.xlarge',
    instance_count=3,
    sagemaker_session=session
)

# Run Spark job
spark_processor.run(
    submit_app='spark_etl.py',
    inputs=[
        ProcessingInput(
            source='s3://my-bucket/raw-data/',
            destination='/opt/ml/processing/input'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination='s3://my-bucket/processed-data/'
        )
    ]
)

## FrameworkProcessor for TensorFlow/PyTorch

In [ ]:
from sagemaker.processing import FrameworkProcessor

tf_processor = FrameworkProcessor(
    estimator_cls=TensorFlow,
    framework_version='2.8',
    role=role,
    instance_type='ml.p3.2xlarge',
    instance_count=1,
    sagemaker_session=session
)

# Run TensorFlow processing job
tf_processor.run(
    code='data_augmentation.py',
    inputs=[
        ProcessingInput(
            source='s3://my-bucket/images/',
            destination='/opt/ml/processing/input'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination='s3://my-bucket/augmented-images/'
        )
    ]
)

## Custom Processing Container

```bash
# Dockerfile for custom processor
FROM python:3.9

RUN pip install pandas numpy scikit-learn

COPY processing_script.py /opt/ml/code/processing_script.py

ENTRYPOINT ["python", "/opt/ml/code/processing_script.py"]
```

## Processing Job Configuration

```json
{
  "processing_job_config": {
    "job_name": "data-prep-job",
    "role_arn": "arn:aws:iam::123456789012:role/SageMakerRole",
    "processing_inputs": [
      {
        "input_name": "input-1",
        "s3_input": {
          "s3_uri": "s3://my-bucket/raw-data/",
          "local_path": "/opt/ml/processing/input",
          "s3_data_type": "S3Prefix",
          "s3_input_mode": "File"
        }
      }
    ],
    "processing_output_config": {
      "outputs": [
        {
          "output_name": "output-1",
          "s3_output": {
            "s3_uri": "s3://my-bucket/processed-data/",
            "local_path": "/opt/ml/processing/output",
            "s3_upload_mode": "EndOfJob"
          }
        }
      ]
    },
    "processing_resources": {
      "cluster_config": {
        "instance_count": 1,
        "instance_type": "ml.m5.xlarge",
        "volume_size_in_gb": 30
      }
    }
  }
}
```

## Monitoring Processing Jobs

In [ ]:
# Check processing job status
import boto3

sm_client = boto3.client('sagemaker')

response = sm_client.describe_processing_job(
    ProcessingJobName='data-prep-job-2024-01-15-12-30-45'
)

print(f"Status: {response['ProcessingJobStatus']}")
print(f"Exit code: {response['ExitCode']}")
print(f"Logs: {response['ProcessingOutputConfig']}")

## Quiz 1

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is the primary purpose of SageMaker Processing Jobs?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q2847391" value="0">
      <span>Large-scale data preparation and ETL</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q2847391" value="1">
      <span>Model training</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q2847391" value="2">
      <span>Real-time predictions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q2847391" value="3">
      <span>Model monitoring</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ Which processor is best for distributed Spark jobs?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384921" value="0">
      <span>SKLearnProcessor</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384921" value="1">
      <span>FrameworkProcessor</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384921" value="2">
      <span>PySparkProcessor</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384921" value="3">
      <span>CustomProcessor</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What does SKLearnProcessor use for data processing?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="0">
      <span>Spark</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="1">
      <span>Scikit-learn and Python</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="2">
      <span>TensorFlow</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="3">
      <span>PyTorch</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="3">
  <p class="font-semibold mb-3">❓ Where do Processing Jobs read input data from?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4829374" value="0">
      <span>Local filesystem</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4829374" value="1">
      <span>DynamoDB</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4829374" value="2">
      <span>RDS</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4829374" value="3">
      <span>S3</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is FrameworkProcessor used for?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="0">
      <span>Processing with TensorFlow, PyTorch, or other frameworks</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="1">
      <span>Training models</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="2">
      <span>Deploying endpoints</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="3">
      <span>Monitoring data drift</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>